# Telco Customer Churn — Preprocessing

Turns the raw dataset into a fully numeric table ready for modeling: drop identifier columns, one-hot encode multi-category features, map binary Yes/No columns to 0/1, and fix the TotalCharges data quality issue found during EDA. Output is written to `Telco-Customer-preprocessing-Churn.csv` and consumed by `03_Modeling.ipynb`.

In [11]:
import pandas as pd   

In [12]:
df = pd.read_csv("../data/Telco-Customer-Churn.csv")

In [13]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Drop identifier columns
`customerID` is unique per row and carries no predictive signal — dropping it (see EDA conclusions).

In [14]:
drop_col = ["customerID" , ]
df.drop(drop_col , inplace=True , axis=1)

In [15]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Encode categorical features
One-hot encode the multi-category columns identified in EDA with `pd.get_dummies`.

**Note for a future iteration:** this currently keeps all category levels (no `drop_first=True`), which creates redundant/collinear dummy columns. It doesn't hurt the tree-based models, but for Logistic Regression it's cleaner to drop one level per column. Not changed here since it would require re-running the full GridSearchCV in `03_Modeling.ipynb` to confirm results don't shift.

In [16]:
categorical_cols =[
       "MultipleLines" ,
       'InternetService',
       'OnlineSecurity',
       'OnlineBackup',
       'DeviceProtection',
       'TechSupport',
       'StreamingTV',
       'StreamingMovies',
       'Contract', 
       'PaymentMethod',
]
df = pd.get_dummies( df ,columns= categorical_cols , dtype=int)

### Encode binary features
Map the Yes/No and Female/Male columns to 0/1 so every feature is numeric.

In [17]:
cols = ["Dependents" , "PaperlessBilling" , "Partner", "PhoneService", "Churn" , ]
for col in cols :
    df[col] =df[col].map({"No" : 0 , "Yes" : 1})
df["gender"] =df["gender"].map({"Female" : 0 , "Male" : 1})

### Fix TotalCharges
`TotalCharges` was stored as text because 11 rows (all `tenure == 0`, brand-new customers) had a blank value instead of a number. Coerce to numeric and fill the resulting NaNs with the median — done on the full dataset here, before the train/test split happens in the modeling notebook.

In [18]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
df.isnull().sum()

gender                                     0
SeniorCitizen                              0
Partner                                    0
Dependents                                 0
tenure                                     0
PhoneService                               0
PaperlessBilling                           0
MonthlyCharges                             0
TotalCharges                               0
Churn                                      0
MultipleLines_No                           0
MultipleLines_No phone service             0
MultipleLines_Yes                          0
InternetService_DSL                        0
InternetService_Fiber optic                0
InternetService_No                         0
OnlineSecurity_No                          0
OnlineSecurity_No internet service         0
OnlineSecurity_Yes                         0
OnlineBackup_No                            0
OnlineBackup_No internet service           0
OnlineBackup_Yes                           0
DeviceProt

In [19]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,0,...,1,0,0,1,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,1,0,0,0,1,0,0,0,0,1
2,1,0,0,0,2,1,1,53.85,108.15,1,...,1,0,0,1,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,1,0,0,0,1,0,1,0,0,0
4,0,0,0,0,2,1,1,70.70,151.65,1,...,1,0,0,1,0,0,0,0,1,0


### Save preprocessed data
Written as a new CSV so `03_Modeling.ipynb` never has to repeat these steps.

In [20]:
df.to_csv("../data/Telco-Customer-preprocessing-Churn.csv" , index=False)